In [ ]:
import os
import json
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials

def download_prefix(client, bucket_name, prefix, local_root):
    bucket = client.bucket(bucket_name)
    blobs = client.list_blobs(bucket_name, prefix=prefix)
    for blob in blobs:
        rel_path = blob.name[len(prefix):]
        if not rel_path:
            continue  
        local_path = os.path.join(local_root, rel_path)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        blob.download_to_filename(local_path)


def main():
    bucket_name = "crfm-helm-public"
    base_prefix = "lite/benchmark_output"
    release = "v1.13.0"
    local_base = os.path.abspath("../helm_lite_v1.13.0")

    client = storage.Client(credentials=AnonymousCredentials(), project="")

    release_prefix = f"{base_prefix}/releases/{release}/"
    local_release_dir = os.path.join(local_base, "releases", release)
    os.makedirs(local_release_dir, exist_ok=True)
    download_prefix(client, bucket_name, release_prefix, local_release_dir)

    summary_path = os.path.join(local_release_dir, "summary.json")
    if not os.path.isfile(summary_path):
        print(f"Error: summary.json not found at {summary_path}")
        return
    with open(summary_path, "r") as f:
        data = json.load(f)
    suites = data.get("suites", [])

    for suite in suites:
        suite_prefix = f"{base_prefix}/runs/{suite}/"
        local_run_dir = os.path.join(local_base, "runs", suite)
        os.makedirs(local_run_dir, exist_ok=True)
        print(f"Downloading suite '{suite}' from {suite_prefix}...")
        download_prefix(client, bucket_name, suite_prefix, local_run_dir)

if __name__ == "__main__":
    main()